<a href="https://colab.research.google.com/github/kinjalkothari/socialintelligence/blob/main/02_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

DATA_PATH = "/LegitReachResearchD2CDataFashionAndApparel.csv"
df = pd.read_csv(DATA_PATH)

df.shape


(14288, 43)

In [3]:
drop_full_null_cols = ["whatsapp", "Snapchat", "yelp", "vimeo"]
df = df.drop(columns=drop_full_null_cols)

df.shape


(14288, 39)

Columns with 100% missing values were removed, as they provide no variance or predictive signal.

In [4]:
numeric_like_cols = [
    "product_page_count",
    "variant_count",
    "average_product_price"
]

for col in numeric_like_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df[numeric_like_cols].dtypes


,0
product_page_count,float64
variant_count,float64
average_product_price,float64


In [5]:
social_cols = [
    col for col in df.columns
    if any(
        kw in col.lower()
        for kw in ["instagram", "facebook", "twitter", "linkedin",
                   "tiktok", "youtube", "pinterest", "x"]
    )
]

df[social_cols] = df[social_cols].notna().astype(int)

df[social_cols].sum().sort_values(ascending=False)


,0
instagram,12533
facebook,10015
tiktok,2730
twitter,2391
pinterest,2283
youtube,1509
linkedin,631
x,114


Missing values in social platform columns are treated as absence or inactivity (0), while presence is encoded as 1. This preserves the semantic meaning of platform adoption.


In [6]:
num_cols = df.select_dtypes(include=[np.number]).columns

for col in num_cols:
    if df[col].isnull().mean() < 0.3:
        df[col] = df[col].fillna(df[col].median())


In [7]:
cat_cols = df.select_dtypes(include=["object"]).columns

df[cat_cols] = df[cat_cols].fillna("unknown")


Features with extreme missingness were intentionally left unimputed or handled via exclusion in later stages to avoid introducing artificial signal.


In [9]:
df.isnull().mean().sort_values(ascending=False).head(10)
df.shape
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14288 entries, 0 to 14287
Data columns (total 39 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   website                  14288 non-null  object 
 1   company                  14288 non-null  object 
 2   country                  14288 non-null  object 
 3   state                    14288 non-null  object 
 4   platform                 14288 non-null  object 
 5   technology_spend         14288 non-null  float64
 6   sales_revenue            14288 non-null  int64  
 7   estimated_visits         14288 non-null  float64
 8   website_created_at       14288 non-null  object 
 9   emails                   14288 non-null  object 
 10  telephones               14288 non-null  object 
 11  vertical                 14288 non-null  object 
 12  employees                33 non-null     float64
 13  product_page_count       14288 non-null  float64
 14  variant_count         

In [10]:
df.to_csv("../data_preprocessed.csv", index=False)
